# 07 · Train the 3D tumor segmenter on Kaggle

All code lives in the `neurolens_ml` package; this notebook only runs it on a Kaggle GPU.

1. Clone the repo and install the package (+ MONAI).
2. Run `python -m neurolens_ml.train_seg` with `ml/configs/seg_segresnet.yaml`: SegResNet on BraTS 2021, random 128³ patches, sliding-window val Dice (TC, WT, ET) every 2 epochs.
3. Show the val Dice curves and the hardest val cases.

**Kaggle setup:** GPU T4 x2 (one GPU is used), Internet on, input `dschettler8845/brats-2021-task1`. Run with *Save Version → Save & Run All*; it stops by itself before the 12-hour limit. Download everything from *Output*.

The first epoch is slow: each scan is unzipped, cropped and normalized once and cached on local disk (~40 GB in `/tmp`). Later epochs only read the cache, so the GPU is no longer waiting on Kaggle's 4 CPUs.

The test split is never opened here (tune on val only).

In [ ]:
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sanjardevx/neurolens.git"
BRANCH = "develop"
CONFIG = "ml/configs/seg_segresnet.yaml"
REPO_DIR = Path(os.environ.get("NEUROLENS_REPO_DIR", "/tmp/neurolens"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
EXTRA_ARGS = os.environ.get("NEUROLENS_ARGS", "").split()  # only for local smoke tests (fake data paths, --smoke)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
# --no-deps: keep Kaggle's own torch and numpy; MONAI and nibabel need nothing else
missing = [p for p in ["neurolens_ml", "monai", "nibabel"] if importlib.util.find_spec(p) is None]
if "neurolens_ml" in missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR / "ml")], check=True)
if pkgs := [p for p in missing if p != "neurolens_ml"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *pkgs], check=True)

import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("repo commit:", commit, "| GPU:", gpu, "| CPUs:", os.cpu_count(), "| free disk in /tmp:", shutil.disk_usage("/tmp").free // 10**9, "GB")

In [ ]:
cmd = [sys.executable, "-m", "neurolens_ml.train_seg", "--config", str(REPO_DIR / CONFIG), "--out", str(OUT_DIR), *EXTRA_ARGS]
print(" ".join(cmd))
# Stream the training log into the notebook output, line by line
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "training failed, see the log above"

In [ ]:
import pandas as pd
import yaml
from IPython.display import Image, display

name = yaml.safe_load((REPO_DIR / CONFIG).read_text())["name"]
metrics = json.loads((OUT_DIR / f"metrics_seg_{name}.json").read_text())
metrics["repo_commit"] = commit  # which code made this model
metrics["gpu"] = gpu
(OUT_DIR / f"metrics_seg_{name}.json").write_text(json.dumps(metrics, indent=2))
print("best epoch:", metrics["best_epoch"], "of", metrics["epochs_run"], "|", metrics["hours"], "h | val Dice:", metrics["val"])
display(Image(filename=str(OUT_DIR / f"curves_seg_{name}.png")))
cases = pd.read_csv(OUT_DIR / f"val_cases_seg_{name}.csv")
cases.assign(mean=cases[["tc", "wt", "et"]].mean(axis=1)).sort_values("mean").head(10).round(3)  # hardest val cases